# Step 1 & 2. 데이터 전처리하기
영어와 달리 한국어의 특성을 반영하여 한글, 영문, 숫자, 기본 구두점만 남기고 제거하는 전처리를 수행합니다.

In [20]:
import pandas as pd
import re
import os
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 데이터 로드 (user_data.csv라는 파일로 저장되어 있다고 가정)
df = pd.read_csv('work\songys_chatbot\ChatbotData.csv')
# 아래는 제공해주신 예시 데이터입니다.
# data = {
#     'Q': ['12시 땡!', '1지망 학교 떨어졌어', '3박4일 놀러가고 싶다', '3박4일 정도 놀러가고 싶다', 'PPL 심하네', 'SD카드 망가졌어'],
#     'A': ['하루가 또 가네요.', '위로해 드립니다.', '여행은 언제나 좋죠.', '여행은 언제나 좋죠.', '눈살이 찌푸려지죠.', '다시 새로 사는 게 마음 편해요.'],
#     'label': [0, 0, 0, 0, 0, 0]
# }
# df = pd.DataFrame(data)

def preprocess_sentence(sentence):
    # 양쪽 공백 제거
    sentence = sentence.strip()
    
    # 구두점 양쪽에 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    
    # 한글, 알파벳, 숫자, 구두점(?, ., !, ,)을 제외한 모든 문자를 공백으로 대체
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    
    return sentence.strip()

# 전처리 적용
df['Q'] = df['Q'].apply(preprocess_sentence)
df['A'] = df['A'].apply(preprocess_sentence)
print("전처리 완료된 데이터 확인:\n", df.head())

전처리 완료된 데이터 확인:
                  Q             A  label
0          12시 땡 !   하루가 또 가네요 .      0
1      1지망 학교 떨어졌어    위로해 드립니다 .      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠 .      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠 .      0
4          PPL 심하네   눈살이 찌푸려지죠 .      0


<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
C:\Users\H11\AppData\Local\Temp\ipykernel_32364\1855592158.py:10: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv('work\songys_chatbot\ChatbotData.csv')


# Step 3. SentencePiece 사용하기
형태소 분석기 대신 제공해주신 텍스트 데이터를 바탕으로 SentencePiece의 BPE(Byte-Pair Encoding) 모델을 학습합니다.

In [21]:
# SentencePiece 학습을 위한 텍스트 파일 생성
corpus_file = "korean_chatbot_corpus.txt"
with open(corpus_file, 'w', encoding='utf-8') as f:
    for q, a in zip(df['Q'], df['A']):
        f.write(q + "\n")
        f.write(a + "\n")

# SentencePiece 모델 학습
vocab_size = 8000 # 실제 데이터가 적을 경우 이 값을 100~500 등으로 낮춰야 합니다.
spm.SentencePieceTrainer.Train(
    input=corpus_file,
    model_prefix="spm_korean",
    # vocab_size=len(set(" ".join(df['Q'] + " " + df['A']).split())) + 10, # 예제용 임시 크기
    vocab_size=5000, # 동적 계산 대신 고정값 사용
    character_coverage=0.9995,
    model_type="bpe",
    max_sentence_length=9999,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

# 학습된 모델 로드
sp = spm.SentencePieceProcessor()
sp.Load("spm_korean.model")

print("토크나이징 테스트:", sp.encode_as_pieces("12시 땡 ! 하루가 또 가네요 ."))

토크나이징 테스트: ['▁1', '2', '시', '▁땡', '▁!', '▁하루', '가', '▁또', '▁가', '네요', '▁.']


# Step 4. 모델 구성하기
전처리 및 토크나이징된 데이터를 파이토치 Dataset으로 만들고, 이전 실습에서 사용한 트랜스포머 모델을 그대로 연결합니다.

In [22]:
import torch
import torch.nn as nn
import math

# ==========================================
# 1. 마스크(Mask) 생성 함수
# ==========================================
def create_padding_mask(x, pad_id=0):
    # 패딩(pad_id)이 있는 위치를 1로 마스킹
    mask = (x == pad_id).float()
    return mask.unsqueeze(1).unsqueeze(2)  # (batch_size, 1, 1, seq_len)

def create_look_ahead_mask(x):
    # 디코더에서 미래의 단어를 보지 못하도록 가리는 마스크
    seq_len = x.size(1)
    look_ahead_mask = 1 - torch.tril(torch.ones(seq_len, seq_len)).type_as(x.float())
    return look_ahead_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

# ==========================================
# 2. 포지셔널 인코딩 (Positional Encoding)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(max_len, d_model)

    def get_angles(self, position, i, d_model):
        angles = 1 / torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        return position * angles

    def positional_encoding(self, max_len, d_model):
        angle_rads = self.get_angles(
            position=torch.arange(max_len, dtype=torch.float32).unsqueeze(1),
            i=torch.arange(d_model, dtype=torch.float32).unsqueeze(0),
            d_model=d_model
        )
        sines = torch.sin(angle_rads[:, 0::2])
        cosines = torch.cos(angle_rads[:, 1::2])
        
        pos_encoding = torch.zeros(angle_rads.shape)
        pos_encoding[:, 0::2] = sines
        pos_encoding[:, 1::2] = cosines
        pos_encoding = pos_encoding.unsqueeze(0) # (1, max_len, d_model)
        return pos_encoding

    def forward(self, x):
        return x + self.pos_encoding[:, :x.size(1), :].to(x.device)

# ==========================================
# 3. 멀티-헤드 어텐션 (Multi-Head Attention)
# ==========================================
def scaled_dot_product_attention(query, key, value, mask):
    matmul_qk = torch.matmul(query, key.transpose(-2, -1))
    d_k = query.size(-1)
    logits = matmul_qk / math.sqrt(d_k)
    
    if mask is not None:
        logits += (mask * -1e9)  # 마스킹된 위치는 매우 작은 값으로 처리
        
    attention_weights = torch.softmax(logits, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0
        self.depth = d_model // self.num_heads
        
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.transpose(1, 2)

    def forward(self, q, k, v, mask):
        batch_size = q.size(0)
        
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)
        
        scaled_attention = scaled_dot_product_attention(q, k, v, mask)
        scaled_attention = scaled_attention.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        output = self.dense(scaled_attention)
        return output

# ==========================================
# 4. 인코더 및 디코더 레이어
# ==========================================
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.mha(x, x, x, mask)
        out1 = self.layernorm1(x + self.dropout1(attn_output))
        ffn_output = self.ffn(out1)
        out2 = self.layernorm2(out1 + self.dropout2(ffn_output))
        return out2

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.layernorm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        attn1 = self.mha1(x, x, x, look_ahead_mask)
        out1 = self.layernorm1(x + self.dropout1(attn1))
        
        attn2 = self.mha2(out1, enc_output, enc_output, padding_mask)
        out2 = self.layernorm2(out1 + self.dropout2(attn2))
        
        ffn_output = self.ffn(out2)
        out3 = self.layernorm3(out2 + self.dropout3(ffn_output))
        return out3

# ==========================================
# 5. 인코더 및 디코더 블록
# ==========================================
class Encoder(nn.Module):
    def __init__(self, vocab_size, num_layers, ff_dim, d_model, num_heads, dropout=0.1, max_len=10000):
        super(Encoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(max_len, d_model)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        for i in range(self.num_layers):
            x = self.enc_layers[i](x, mask)
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, num_layers, ff_dim, d_model, num_heads, dropout=0.1, max_len=10000):
        super(Decoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(max_len, d_model)
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        for i in range(self.num_layers):
            x = self.dec_layers[i](x, enc_output, look_ahead_mask, padding_mask)
        return x

# ==========================================
# 6. 트랜스포머 메인 모델
# ==========================================
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, units, d_model, num_heads, dropout=0.1, pad_id=0):
        super(Transformer, self).__init__()
        self.pad_id = pad_id
        self.encoder = Encoder(vocab_size, num_layers, units, d_model, num_heads, dropout)
        self.decoder = Decoder(vocab_size, num_layers, units, d_model, num_heads, dropout)
        self.final_layer = nn.Linear(d_model, vocab_size)

    def forward(self, enc_inputs, dec_inputs):
        # 마스크 생성
        enc_padding_mask = create_padding_mask(enc_inputs, self.pad_id)
        dec_padding_mask = create_padding_mask(enc_inputs, self.pad_id)
        
        # 디코더의 look_ahead_mask는 패딩 마스크와 합쳐집니다
        look_ahead_mask = torch.max(
            create_padding_mask(dec_inputs, self.pad_id),
            create_look_ahead_mask(dec_inputs)
        )

        # 인코더 및 디코더 통과
        enc_output = self.encoder(enc_inputs, enc_padding_mask)
        dec_output = self.decoder(dec_inputs, enc_output, look_ahead_mask, dec_padding_mask)
        
        # 최종 출력
        final_output = self.final_layer(dec_output)
        return final_output

# ==========================================
# 7. 모델 생성 및 옵티마이저 설정
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# SentencePiece의 패딩 ID 확인 (일반적으로 0입니다)
PAD_ID = sp.pad_id()

# 트랜스포머 모델 인스턴스화
# model = Transformer(
#     vocab_size=sp.GetPieceSize(), 
#     num_layers=2, 
#     units=512, 
#     d_model=256, 
#     num_heads=8,
#     pad_id=PAD_ID
# ).to(device)

# # 옵티마이저 및 손실함수
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# 모델 인스턴스화 수정
model = Transformer(
    vocab_size=sp.GetPieceSize(), 
    num_layers=2, 
    units=512, 
    d_model=256, 
    num_heads=4,   # 8에서 4로 수정
    pad_id=PAD_ID
).to(device)

# 학습률 수정
# 기존: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

loss_function = nn.CrossEntropyLoss(ignore_index=PAD_ID)

print("✅ 독립적인 트랜스포머 모델 구성이 완료되었습니다!")

✅ 독립적인 트랜스포머 모델 구성이 완료되었습니다!


In [23]:
class ChatbotDataset(Dataset):
    def __init__(self, df, sp, max_length=40):
        self.data = []
        for _, row in df.iterrows():
            q_ids = sp.EncodeAsIds(row['Q'])
            a_ids = sp.EncodeAsIds(row['A'])

            bos_id = sp.bos_id()
            eos_id = sp.eos_id()

            # 특수 토큰 추가
            q_tokens = [bos_id] + q_ids + [eos_id]
            a_tokens = [bos_id] + a_ids + [eos_id]

            if len(q_tokens) > max_length or len(a_tokens) > max_length:
                continue

            # 패딩(Padding)
            q_tokens += [sp.pad_id()] * (max_length - len(q_tokens))
            a_tokens += [sp.pad_id()] * (max_length - len(a_tokens))

            # Teacher Forcing: 디코더 입력은 마지막 토큰 제외, 타겟은 첫 번째 토큰 제외
            dec_input = a_tokens[:-1]
            target = a_tokens[1:]

            self.data.append({
                "enc_input": q_tokens,
                "dec_input": dec_input,
                "target": target
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        return (torch.tensor(sample["enc_input"], dtype=torch.long),
                torch.tensor(sample["dec_input"], dtype=torch.long),
                torch.tensor(sample["target"], dtype=torch.long))

dataset = ChatbotDataset(df, sp, max_length=40)
#dataloader = DataLoader(dataset, batch_size=2, shuffle=True)# 현재 DataLoader가 batch_size=2로 설정되어 있습니다. 트랜스포머는 한 번에 너무 적은 데이터를 보고 가중치를 업데이트하면 그래디언트(기울기) 변동이 극심해져서 학습 방향을 잡지 못하고 Loss가 크게 요동치게 됩니다.
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# -------------------------------------------------------------
# 트랜스포머 초기화 (이전 실습에서 구현하신 Transformer 클래스 호출)
# model = Transformer(vocab_size=sp.GetPieceSize(), num_layers=2, units=512, d_model=256, num_heads=8)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# loss_function = nn.CrossEntropyLoss(ignore_index=sp.pad_id())
# -------------------------------------------------------------

In [24]:
# 트랜스포머 초기화 (이전 실습에서 구현하신 Transformer 클래스 호출)
model = Transformer(vocab_size=sp.GetPieceSize(), num_layers=2, units=512, d_model=256, num_heads=8)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_function = nn.CrossEntropyLoss(ignore_index=sp.pad_id())

# Step 5 & 6. 모델 학습 및 평가하기 (전체 코드)
모델이 학습된 후, 한국어 전처리를 거쳐 대답을 추론할 수 있는 평가(인퍼런스)용 함수를 작성합니다.

In [25]:
# ==========================================
# 8. 모델 학습 (Training) 함수 정의
# ==========================================
def train_step(model, batch, optimizer, loss_function, device):
    model.train()
    
    # dataloader에서 나온 데이터 분리 (인코더 입력, 디코더 입력, 타겟)
    enc_input, dec_input, target = [x.to(device) for x in batch]

    optimizer.zero_grad()

    # 모델 순전파 (Forward)
    logits = model(enc_input, dec_input)  # 형태: (batch_size, seq_len, vocab_size)

    # 손실 계산을 위해 형태 변경 
    # logits: (batch_size * seq_len, vocab_size), target: (batch_size * seq_len)
    logits_flat = logits.view(-1, logits.size(-1))
    target_flat = target.contiguous().view(-1)

    loss = loss_function(logits_flat, target_flat)

    # 역전파 및 가중치 업데이트 (Backward)
    loss.backward()
    optimizer.step()

    return loss.item()

def train_model(model, dataloader, optimizer, loss_function, num_epochs, device):
    # 🌟 [추가된 부분] 모델을 확실하게 device(GPU)로 보냅니다.
    model = model.to(device)
    
    print("🚀 학습을 시작합니다...")
    for epoch in range(num_epochs):
        total_loss = 0
        for step, batch in enumerate(dataloader):
            loss = train_step(model, batch, optimizer, loss_function, device)
            total_loss += loss

            if (step + 1) % 10 == 0 or step == 0:
                print(f"[Epoch {epoch+1}/{num_epochs}] Step {step+1} - Loss: {loss:.4f}")

        avg_loss = total_loss / len(dataloader)
        print(f"========== Epoch {epoch+1} 완료 | 평균 Loss: {avg_loss:.4f} ==========")


# ==========================================
# 9. 모델 평가 (Inference) 함수 정의
# ==========================================
def evaluate_sentence(model, sentence, sp, device, max_length=40):
    model.eval()
    
    # 1. 입력 문장 전처리 (Step 2에서 정의한 함수 활용)
    # *주의: preprocess_sentence 함수가 메모리에 정의되어 있어야 합니다.
    sentence = preprocess_sentence(sentence)
    
    # 2. 토크나이징 및 텐서 변환
    START_TOKEN = sp.bos_id()
    END_TOKEN = sp.eos_id()
    
    # 인코더 입력 생성: [BOS] + 토큰들 + [EOS]
    enc_input_ids = [START_TOKEN] + sp.encode_as_ids(sentence) + [END_TOKEN]
    enc_input = torch.tensor([enc_input_ids], dtype=torch.long).to(device)
    
    # 3. 디코더 초기 입력 설정 (START_TOKEN으로 시작)
    dec_input = torch.tensor([[START_TOKEN]], dtype=torch.long).to(device)
    
    # 4. 단어 예측 루프 (자기회귀적 예측)
    with torch.no_grad():
        for _ in range(max_length):
            # 모델 예측
            logits = model(enc_input, dec_input)
            
            # 가장 마지막 단어의 예측값 가져오기
            last_word_logits = logits[:, -1, :]
            
            # 가장 확률이 높은 단어 ID 추출
            predicted_id = torch.argmax(last_word_logits, dim=-1).item()
            
            # 예측된 단어가 종료 토큰(END_TOKEN)이면 문장 생성 종료
            if predicted_id == END_TOKEN:
                break
                
            # 예측된 단어를 디코더의 다음 입력으로 이어붙임
            predicted_tensor = torch.tensor([[predicted_id]], dtype=torch.long).to(device)
            dec_input = torch.cat([dec_input, predicted_tensor], dim=1)
    
    # 5. 토큰 ID를 실제 텍스트로 디코딩
    # dec_input 배열에서 첫 번째에 있는 START_TOKEN은 제외하고 디코딩
    output_ids = dec_input.squeeze(0).tolist()[1:]
    predicted_sentence = sp.decode_ids(output_ids)
    
    return predicted_sentence


# ==========================================
# 10. 실제 실행 테스트
# ==========================================
if __name__ == "__main__":
    # 1. 학습 진행 (예시로 20 에폭 진행)
    # 이전 Step에서 정의된 dataloader, optimizer, loss_function, device가 필요합니다.
    EPOCHS = 20
    # 1. Device 설정 확인
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 2. 🌟 모델을 Device(GPU)로 확실히 이동
    model = model.to(device)

    # 3. 학습 시작
    EPOCHS = 20
    train_model(model, dataloader, optimizer, loss_function, num_epochs=EPOCHS, device=device)
    
    print("\n💬 챗봇 테스트를 시작합니다.")
    # 2. 테스트 문장 추론
    test_sentences = [
        "12시 땡!",
        "1지망 학교 떨어졌어",
        "3박4일 놀러가고 싶다",
        "SD카드 망가졌어"
    ]
    
    for q in test_sentences:
        answer = evaluate_sentence(model, q, sp, device)
        print(f"사용자: {q}")
        print(f"챗봇봇: {answer}\n")

🚀 학습을 시작합니다...
[Epoch 1/20] Step 1 - Loss: 8.8629
[Epoch 1/20] Step 10 - Loss: 6.4134
[Epoch 1/20] Step 20 - Loss: 6.1005
[Epoch 1/20] Step 30 - Loss: 5.9983
[Epoch 1/20] Step 40 - Loss: 5.9510
[Epoch 1/20] Step 50 - Loss: 5.7780
[Epoch 1/20] Step 60 - Loss: 5.6059
[Epoch 1/20] Step 70 - Loss: 5.6408
[Epoch 1/20] Step 80 - Loss: 5.4585
[Epoch 1/20] Step 90 - Loss: 5.5236
[Epoch 1/20] Step 100 - Loss: 5.1593
[Epoch 1/20] Step 110 - Loss: 5.4468
[Epoch 1/20] Step 120 - Loss: 5.3793
[Epoch 1/20] Step 130 - Loss: 5.0730
[Epoch 1/20] Step 140 - Loss: 5.1149
[Epoch 1/20] Step 150 - Loss: 5.2175
[Epoch 1/20] Step 160 - Loss: 4.9812
[Epoch 1/20] Step 170 - Loss: 4.9085
[Epoch 1/20] Step 180 - Loss: 5.0536
========== Epoch 1 완료 | 평균 Loss: 5.5439 ==========
[Epoch 2/20] Step 1 - Loss: 4.6811
[Epoch 2/20] Step 10 - Loss: 4.4341
[Epoch 2/20] Step 20 - Loss: 4.3608
[Epoch 2/20] Step 30 - Loss: 4.6509
[Epoch 2/20] Step 40 - Loss: 4.2731
[Epoch 2/20] Step 50 - Loss: 4.4552
[Epoch 2/20] Step 60 - Loss

In [26]:
print("\n💬 챗봇 테스트를 시작합니다.")
# 2. 테스트 문장 추론
test_sentences = [
    "12시 땡!",
    "1지망 학교 떨어졌어",
    "3박4일 놀러가고 싶다",
    "SD카드 망가졌어"
]

for q in test_sentences:
    answer = evaluate_sentence(model, q, sp, device)
    print(f"사용자: {q}")
    print(f"챗봇봇: {answer}\n")


💬 챗봇 테스트를 시작합니다.
사용자: 12시 땡!
챗봇봇: 하루가 또 가네요 .

사용자: 1지망 학교 떨어졌어
챗봇봇: 위로해 드립니다 .

사용자: 3박4일 놀러가고 싶다
챗봇봇: 여행은 언제나 좋죠 .

사용자: SD카드 망가졌어
챗봇봇: 다시 새로 사는 게 마음 편해요 .



트랜스포머 모델을 직접 구현하고 학습시키는 과정에서 Loss가 안정적으로 떨어지지 않고 널뛰는(fluctuating) 현상이 발생하고 있는 것으로 보입니다. 올려주신 코드를 분석해 본 결과, 어텐션 메커니즘이나 인코더/디코더 레이어 구조, 마스킹 처리 등 트랜스포머의 핵심 로직은 논문에 맞게 **매우 정확하게 구현**되어 있습니다.

학습이 제대로 되지 않는 주된 이유는 모델의 논리적 오류가 아니라 **하이퍼파라미터 설정**, 그중에서도 배치 사이즈(Batch Size)와 학습률(Learning Rate)의 문제입니다. 아래 3가지 주요 부분을 수정해 보시기를 권장합니다.

### 1. 배치 사이즈(Batch Size) 확대 (가장 중요)

현재 `DataLoader`가 `batch_size=2`로 설정되어 있습니다. 트랜스포머는 한 번에 너무 적은 데이터를 보고 가중치를 업데이트하면 그래디언트(기울기) 변동이 극심해져서 학습 방향을 잡지 못하고 Loss가 크게 요동치게 됩니다.

* **수정:** `batch_size`를 64 (메모리가 부족하다면 최소 32)로 늘려주세요.

```python
# 기존: dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

```

### 2. 학습률(Learning Rate) 및 어텐션 헤드 조정

Adam 옵티마이저에 `1e-3`의 학습률이 적용되어 있습니다. 웜업(Warm-up) 스케줄러가 없는 트랜스포머 구조에서 0.001은 다소 높은 수치이므로 초기에 모델이 발산할 위험이 있습니다. `1e-4`로 낮춰서 안정적으로 학습을 시작하는 것이 좋습니다.
또한 노트북 파일의 설명란에 적혀있던 체급 조언대로, `d_model=256`일 때는 각 헤드가 64차원을 가질 수 있도록 `num_heads=8`보다는 `4`로 설정하는 것이 구조적으로 훨씬 안정적입니다.

```python
# 모델 인스턴스화 수정
model = Transformer(
    vocab_size=sp.GetPieceSize(), 
    num_layers=2, 
    units=512, 
    d_model=256, 
    num_heads=4,   # 8에서 4로 수정
    pad_id=PAD_ID
).to(device)

# 학습률 수정
# 기존: optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

```

### 3. SentencePiece 단어장 크기(Vocab Size) 최적화

현재 SentencePiece 학습 코드에서 `vocab_size`를 `len(set(" ".join(df['Q'] + " " + df['A']).split())) + 10`으로 동적 할당하셨습니다. 데이터셋(약 11,800개) 전체의 띄어쓰기 기준 단어 수로 단어장 크기를 잡으면 서브워드(Subword) 분리 효과가 떨어지고 임베딩 파라미터가 불필요하게 비대해져 과적합이 발생하기 쉽습니다.

* **수정:** 데이터셋 크기에 맞게 `vocab_size`를 5,000 정도로 고정해서 모델을 가볍게 만들어주세요.

```python
spm.SentencePieceTrainer.Train(
    input=corpus_file,
    model_prefix="spm_korean",
    vocab_size=5000, # 동적 계산 대신 고정값 사용
    character_coverage=0.9995,
    model_type="bpe",
    max_sentence_length=9999,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

```

**💡 참고 (마스킹 로직 관련):**
작성하신 `Transformer` 클래스의 `forward` 메서드를 보면 `enc_padding_mask`와 `dec_padding_mask`가 동일하게 `create_padding_mask(enc_inputs, self.pad_id)`를 호출하고 있습니다. 언뜻 보면 실수 같아 보일 수 있지만, 이는 디코더의 크로스 어텐션(Cross-Attention) 단계에서 인코더 출력(Key, Value)의 패딩을 가려주는 역할을 하므로 논리적으로 **정확히 맞는 동작**입니다. 모델의 마스킹 처리는 완벽하니 안심하셔도 됩니다.

위 세 가지 설정(특히 배치 사이즈와 학습률)을 변경하신 후 훈련을 다시 실행해보시면, 이전과 달리 Loss가 부드럽고 안정적으로 하락하는 것을 확인하실 수 있을 것입니다.